In [1]:
!pip install langchain==0.3.25 langchain-core langchain-groq langchain-community langchain-text-splitters faiss-cpu pypdf huggingface-hub sentence-transformers

In [5]:
!pip install langchain-groq

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "API KEY"

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile")
response = llm.invoke("Say hello in one line")
print(response.content)

Hello.


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

loader = PyPDFLoader(filename)
pages = loader.load()

print(f"Total pages loaded: {len(pages)}")
print(f"\nFirst page preview:\n{pages[0].page_content[:300]}")

Saving resume-Maanusri II-YR.pdf to resume-Maanusri II-YR (1).pdf
Total pages loaded: 2

First page preview:
MAANUSRI J 
Phone no:+91 8870087332 
E-mail: maanusrijagan@gmail.com 
city : coimbatore  
Linkedin - maanusri-jagan-cbe2468 
Github: https://github.com/maanu-sri 
Objective 
AI & Data Science student actively working on hands-on projects and seeking opportunities to 
gain practical experience. 
Proj


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print(f"Total chunks created: {len(chunks)}")
print(f"\nChunk 1 preview:\n{chunks[0].page_content}")

Total chunks created: 5

Chunk 1 preview:
MAANUSRI J 
Phone no:+91 8870087332 
E-mail: maanusrijagan@gmail.com 
city : coimbatore  
Linkedin - maanusri-jagan-cbe2468 
Github: https://github.com/maanu-sri 
Objective 
AI & Data Science student actively working on hands-on projects and seeking opportunities to 
gain practical experience. 
Projects  
Sales Analytics Dashboard  
Developed an interactive sales analytics dashboard using Python and Streamlit to visualize


In [5]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Convert chunks to embeddings and store in FAISS
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

print("✅ Embeddings created and stored in FAISS!")
print(f"Total vectors stored: {vectorstore.index.ntotal}")

/tmp/ipykernel_25054/955973573.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embeddings created and stored in FAISS!
Total vectors stored: 5


In [19]:
!pip install langchain langchain-core langchain-groq langchain-community langchain-text-splitters langchain-google-genai faiss-cpu pypdf streamlit python-dotenv huggingface-hub sentence-transformers

In [8]:
from langchain_groq import ChatGroq
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Load LLM
llm = ChatGroq(model="llama-3.3-70b-versatile")

# Prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below:

Context: {context}

Question: {input}
""")

# Create chains
document_chain = create_stuff_documents_chain(llm, prompt)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retrieval_chain = create_retrieval_chain(retriever, document_chain)

# Ask question
question = "What is her email id?"
result = retrieval_chain.invoke({"input": question})

print(f"Question: {question}")
print(f"\nAnswer: {result['answer']}")

Question: What is her email id?

Answer: Her email ID is maanusrijagan@gmail.com.
